<a href="https://colab.research.google.com/github/ezh2020/FloodRoad-SAM3/blob/main/FloodRoad_SAM3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FloodRoad-SAM3 课程设计 Demo

这是我们的 FloodRoad-SAM3 演示 notebook。核心链路是：准备 SpaceNet 8 样本，按需要从 Google Drive 下载已训练权重，然后直接进行推理、验证和可视化展示。

运行前需要：GPU runtime、Hugging Face `facebook/sam3` 访问权限、Colab Secret `HF_TOKEN`。如果权重链接留空，对应模型会重新训练；如果填写 Google Drive 链接，就下载权重并跳过训练。


## 1. 准备代码和运行环境


In [ ]:
import os
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path

REPO_URL = "https://github.com/ezh2020/FloodRoad-SAM3.git"
REPO_DIR = Path("/content/FloodRoad-SAM3")
VENV_DIR = Path("/content/floodroad_venv")
PYTHON = VENV_DIR / "bin" / "python"
BASE_PYTHON = Path("/usr/local/bin/python3") if Path("/usr/local/bin/python3").exists() else Path(sys.executable)
VIRTUALENV_PYZ = Path("/content/virtualenv.pyz")

def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)), flush=True)
    result = subprocess.run(cmd, text=True, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result

def create_venv():
    print("Base Python for venv:", BASE_PYTHON, flush=True)
    result = subprocess.run([str(BASE_PYTHON), "-m", "venv", "--system-site-packages", str(VENV_DIR)], text=True)
    if result.returncode == 0:
        return
    print("python -m venv failed; using virtualenv.pyz fallback.", flush=True)
    if not VIRTUALENV_PYZ.exists():
        urllib.request.urlretrieve("https://bootstrap.pypa.io/virtualenv.pyz", VIRTUALENV_PYZ)
    run([str(BASE_PYTHON), str(VIRTUALENV_PYZ), "--system-site-packages", str(VENV_DIR)])

%cd /content
shutil.rmtree(REPO_DIR, ignore_errors=True)
shutil.rmtree(VENV_DIR, ignore_errors=True)
run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
%cd /content/FloodRoad-SAM3
!git rev-parse --short HEAD
create_venv()
assert PYTHON.exists(), f"Virtual environment Python was not created at {PYTHON}"
run([str(PYTHON), "--version"])
run([str(PYTHON), "-m", "pip", "install", "-q", "--no-warn-conflicts", "--upgrade", "pip", "setuptools", "wheel"])
run([str(PYTHON), "-m", "pip", "install", "-q", "--no-warn-conflicts", "-r", "requirements.txt"])
run([str(PYTHON), "-m", "py_compile", "run_colab_real.py", "train.py", "evaluate.py", "data/preprocess_sn8.py", "data/preprocess.py", "data/dataset.py", "models/sam3_baseline.py", "models/floodroad_sam3.py", "models/vit_token_merging.py", "models/dca.py", "models/rgstm.py", "models/ccrl.py", "models/deeplabv3_baseline.py", "models/losses.py", "models/lora.py", "metrics.py", "utils.py", "colab_validate.py"])
os.environ["FLOODROAD_PYTHON"] = str(PYTHON)
os.environ["PATH"] = str(PYTHON.parent) + os.pathsep + os.environ.get("PATH", "")
print("Using isolated Python:", os.environ["FLOODROAD_PYTHON"])


## 2. 加载 SAM3 权重访问凭证


In [ ]:
import os
from google.colab import userdata

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or userdata.get("HF_TOKEN")
assert token, "Missing Colab Secret named HF_TOKEN. Add it after your Hugging Face account has access to facebook/sam3."
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token
print("HF token loaded:", bool(token))


## 3. 检查数据、依赖和 SAM3 接口


In [ ]:
import os
import json
import subprocess
import sys
from pathlib import Path

validation_path = Path("/content/floodroad_runs/default/colab_validation.json")
validation_path.parent.mkdir(parents=True, exist_ok=True)
python_bin = os.environ.get("FLOODROAD_PYTHON", sys.executable)
cmd = [python_bin, "colab_validate.py", "--output", str(validation_path)]
print("+", " ".join(cmd), flush=True)
env = dict(os.environ)
env["PATH"] = str(Path(python_bin).parent) + os.pathsep + env.get("PATH", "")
subprocess.run(cmd, text=True, check=True, env=env)
validation = json.loads(validation_path.read_text())
os.environ["SN8_TARBALL"] = validation["sn8_tarball"]
print("SN8_TARBALL=", os.environ["SN8_TARBALL"])
print("SAM3 hits=", validation["sam3_hits"])
print("Validation metadata=", validation_path)


## 4. 下载权重并运行推理验证


In [ ]:
# DeepLab 权重链接留空时会重新训练；填入链接时下载权重并跳过训练。
DEEPLAB_CHECKPOINT_URL = ""

# Ours 权重链接留空时会重新训练；填入链接时下载权重并跳过对应训练。
# 这里默认调用已训练权重下载链接进行推理和验证。
# 第一个链接是 ours_no_tm.pt；第二个链接是 ours_tm.pt。
OURS_NO_TM_CHECKPOINT_URL = "https://drive.google.com/file/d/1w2zF_a8yXF9JaaAlyCSI8Z718mCQ3ytx/view?usp=drive_link"
OURS_TM_CHECKPOINT_URL = "https://drive.google.com/file/d/1oVdnWXi7QAkMNh10TajnpE0S5wgetsfC/view?usp=drive_link"

# Leave source-domain fields empty to train DeepLab on the current Louisiana-East data.
DEEPLAB_SOURCE_LOCATION = ""
DEEPLAB_SOURCE_TARBALL = ""
DEEPLAB_SOURCE_LIMIT_RECORDS = 32

import os
import subprocess
os.environ["DEEPLAB_CHECKPOINT_URL"] = DEEPLAB_CHECKPOINT_URL.strip()
os.environ["OURS_NO_TM_CHECKPOINT_URL"] = OURS_NO_TM_CHECKPOINT_URL.strip()
os.environ["OURS_TM_CHECKPOINT_URL"] = OURS_TM_CHECKPOINT_URL.strip()

# Restore the adapter file from the checked-out repo state in case an earlier hot patch left bad indentation.
subprocess.run(["git", "checkout", "HEAD", "--", "models/sam3_baseline.py"], check=True)
subprocess.run([os.environ["FLOODROAD_PYTHON"], "-m", "py_compile", "models/sam3_baseline.py", "models/vit_token_merging.py", "run_colab_real.py", "train.py", "evaluate.py", "visualize_segments.py"], check=True)

help_probe = subprocess.run(
    [os.environ["FLOODROAD_PYTHON"], "run_colab_real.py", "--help"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
if "--ours-no-tm-checkpoint-url" not in help_probe.stdout:
    print(help_probe.stdout)
    raise RuntimeError("run_colab_real.py is older than this notebook. Re-run section 1 to clone the latest repo, or run `git pull` in /content/FloodRoad-SAM3.")

cmd = [
    os.environ["FLOODROAD_PYTHON"], "run_colab_real.py",
    "--config", "configs/default.yaml",
    "--raw-root", "/content/spacenet8/raw",
    "--processed-root", "/content/spacenet8/processed",
    "--output-dir", "/content/floodroad_runs/default",
    "--sn8-tarball", os.environ["SN8_TARBALL"],
    "--skip-sam3-install",
]
if DEEPLAB_CHECKPOINT_URL.strip():
    cmd += ["--deeplab-checkpoint-url", DEEPLAB_CHECKPOINT_URL.strip()]
if OURS_NO_TM_CHECKPOINT_URL.strip():
    cmd += ["--ours-no-tm-checkpoint-url", OURS_NO_TM_CHECKPOINT_URL.strip()]
if OURS_TM_CHECKPOINT_URL.strip():
    cmd += ["--ours-tm-checkpoint-url", OURS_TM_CHECKPOINT_URL.strip()]
if not DEEPLAB_CHECKPOINT_URL.strip() and DEEPLAB_SOURCE_LOCATION.strip():
    cmd += ["--deeplab-source-location", DEEPLAB_SOURCE_LOCATION.strip()]
if not DEEPLAB_CHECKPOINT_URL.strip() and DEEPLAB_SOURCE_TARBALL.strip():
    cmd += ["--deeplab-source-tarball", DEEPLAB_SOURCE_TARBALL.strip()]
if not DEEPLAB_CHECKPOINT_URL.strip() and (DEEPLAB_SOURCE_LOCATION.strip() or DEEPLAB_SOURCE_TARBALL.strip()):
    cmd += [
        "--deeplab-source-raw-root", "/content/spacenet8/deeplab_source_raw",
        "--deeplab-source-processed-root", "/content/spacenet8/deeplab_source_processed",
        "--deeplab-source-limit-records", str(DEEPLAB_SOURCE_LIMIT_RECORDS),
    ]
print("+", " ".join(cmd), flush=True)
from pathlib import Path
log_path = Path("/content/floodroad_runs/default/run_colab_real_latest.log")
log_path.parent.mkdir(parents=True, exist_ok=True)
result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
log_path.write_text(result.stdout)
print(result.stdout[-20000:])
print("Full log:", log_path)
if result.returncode != 0:
    raise RuntimeError(f"run_colab_real.py failed with exit code {result.returncode}; see {log_path}")


## 5. 展示验证结果


In [ ]:
from pathlib import Path
from IPython.display import Markdown, display

out = Path("/content/floodroad_runs/default")
for name, title in [("accuracy_table.md", "精度对比"), ("efficiency_table.md", "效率对比")]:
    path = out / name
    print(f"\n===== {title}: {path} =====")
    if path.exists():
        display(Markdown(path.read_text()))
    else:
        print("missing")

print("\n===== Demo 输出文件 =====")
for path in [
    out / "checkpoints" / "deeplab.pt",
    out / "checkpoints" / "ours_no_tm.pt",
    out / "checkpoints" / "ours_tm.pt",
    out / "accuracy_table.md",
    out / "efficiency_table.md",
    out / "segment_visualizations",
]:
    print(path, "exists" if path.exists() else "missing")


## 6. 单样本道路级打分展示


In [ ]:
# SAMPLE_INDEX=1 表示 rl_samples.json 中第 1 个固定展示样本。
# 默认使用前 20 个 train-split tile 作为 demo 推理/验证样本，可改成 1 到 20。
SAMPLE_INDEX = 1

import json
import os
import subprocess
import sys
from pathlib import Path
from IPython.display import Image, display, Markdown

summary_path = Path('/content/floodroad_runs/default/segment_visualizations/latest_sample.json')
cmd = [
    os.environ.get('FLOODROAD_PYTHON', sys.executable), 'visualize_segments.py',
    '--config', '/content/floodroad_runs/default/real_run.yaml',
    '--output-dir', '/content/floodroad_runs/default',
    '--sample-index', str(SAMPLE_INDEX),
    '--summary-json', str(summary_path),
]
print('+', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)
summary = json.loads(summary_path.read_text())
print('sample', summary['sample_index'], 'of', summary['sample_count'], 'id=', summary['sample_id'])
display(Image(filename=summary['figure']))
display(Markdown(Path(summary['markdown']).read_text()))
